# Batch inference to CSV

The studio predicts one image at a time because that is what a human reviewing
results wants. This is the other half: every item in a split, one row each, on
disk as an artifact you can hand to someone who does not use Orinth.

In [ ]:
import orinth

MODEL_ID = next(model.id for model in orinth.models.list() if model.available)
DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"object_detection", "segmentation", "classification"}
    and dataset.splits.get("test")
)
SPLIT = "test"

predictor = orinth.models.predictor(MODEL_ID)
params = orinth.models.parameters(confidence_threshold=0.5)
paths = orinth.datasets.paths(DATASET_ID, SPLIT)
print(len(paths), "items in", DATASET_ID, SPLIT)

## One row per detection

Flat, not nested: a CSV with a JSON blob in a column is a file nobody can filter.
An item with no detections still gets a row, because "found nothing" is an
answer and dropping it silently changes the denominator of every rate computed
downstream.

In [ ]:
import time

records, started = [], time.perf_counter()
for position, path in enumerate(paths, start=1):
    detections = predictor.predict(path, params)
    if not detections:
        records.append({"item": path.name, "label": None, "confidence": None, "x": None, "y": None, "w": None, "h": None})
        continue
    for detection in detections:
        box = detection.bbox
        records.append(
            {
                "item": path.name,
                "label": detection.class_name,
                "confidence": float(detection.confidence),
                "x": round(box.x, 1),
                "y": round(box.y, 1),
                "w": round(box.width, 1),
                "h": round(box.height, 1),
            }
        )
    if position % 25 == 0:
        print(f"{position}/{len(paths)} · {position / (time.perf_counter() - started):.1f} items/s")

elapsed = time.perf_counter() - started
print(f"\n{len(paths)} items in {elapsed:.1f}s · {len(records)} rows")

## What came out

Two things to look at before trusting the file: the share of items that found
nothing, and the confidence distribution. A pile of predictions just above the
threshold means the threshold is doing the deciding, not the model.

In [ ]:
import polars as pl

predictions = pl.DataFrame(records)
empty = predictions.filter(pl.col("label").is_null()).height
print(f"{empty}/{predictions['item'].n_unique()} items with no detection above threshold\n")

print(predictions.group_by("label").agg(
    pl.len().alias("detections"), pl.col("confidence").mean().alias("mean_confidence")
).sort("detections", descending=True))

In [ ]:
with orinth.runs.start(
    f"batch {MODEL_ID}",
    params={"model": MODEL_ID, "split": SPLIT, "confidence": 0.5},
    dataset_id=DATASET_ID,
) as run:
    run.log(items=float(len(paths)), detections=float(predictions.height), seconds=float(elapsed))
    artifact = run.log_artifact(predictions.write_csv().encode("utf-8"), name="predictions.csv")
    print("wrote", artifact)

The artifact is downloadable from the run in the rail. Nothing left the machine
to produce it.